In [2]:
import os
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# 1. THIẾT LẬP ĐƯỜNG DẪN CẤU TRÚC CHUẨN
# Tự động trỏ vào thư mục exps_/data và exps_/results
base_dir = os.path.abspath(os.path.join(os.getcwd(), "../../"))
DATA_DIR = os.path.join(base_dir, "exps_", "data", "feature1")
RES_DIR = os.path.join(base_dir, "exps_", "results")
os.makedirs(RES_DIR, exist_ok=True)

print("Thư mục chứa dữ liệu:", DATA_DIR)
print("Thư mục lưu kết quả:", RES_DIR)

# 2. ĐỌC DỮ LIỆU ĐÃ QUA TIỀN XỬ LÝ
x_train = pd.read_excel(os.path.join(DATA_DIR, "x_train.xlsx")).fillna(0)
y_train = pd.read_excel(os.path.join(DATA_DIR, "y_train.xlsx"))["SalePrice"]
x_test = pd.read_excel(os.path.join(DATA_DIR, "x_test.xlsx")).fillna(0)

print("Kích thước X_train:", x_train.shape, "| y_train:", y_train.shape)
print("Kích thước X_test:", x_test.shape)

# Chuyển sang dạng Tensor của PyTorch
X_tensor = torch.tensor(x_train.values, dtype=torch.float32)
y_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(x_test.values, dtype=torch.float32)

# Chia tập Train / Validation (80/20)
X_tr, X_va, y_tr, y_va = train_test_split(X_tensor, y_tensor, test_size=0.2, random_state=42)

train_dataset = TensorDataset(X_tr, y_tr)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# 3. ĐỊNH NGHĨA KIẾN TRÚC MẠNG PYTORCH MLP
class HousePriceMLP(nn.Module):
    def __init__(self, input_dim):
        super(HousePriceMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
        
    def forward(self, x):
        return self.network(x)

input_dim = x_train.shape[1]
model = HousePriceMLP(input_dim)

# Thiết lập hàm mất mát (MSE) và bộ tối ưu (Adam)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. HUẤN LUYỆN MÔ HÌNH (TRAINING)
epochs = 50
print("\nBắt đầu huấn luyện mô hình PyTorch MLP...")
for epoch in range(epochs):
    model.train()
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        
    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            val_preds = model(X_va)
            val_loss = criterion(val_preds, y_va)
            val_rmse = np.sqrt(val_loss.item())
        print(f"Epoch [{epoch+1}/{epochs}] - Val RMSE: {val_rmse:.4f}")

# 5. DỰ ĐOÁN TẬP TEST VÀ LƯU KẾT QUẢ ĐÚNG CHUẨN
model.eval()
with torch.no_grad():
    test_preds_log = model(X_test_tensor).numpy().flatten()

# Nếu nhãn y_train lúc đầu bạn dùng log (ví dụ log1p), nhớ đổi ngược lại bằng np.expm1
# test_preds = np.expm1(test_preds_log) 
test_preds = test_preds_log # Dùng trực tiếp nếu y_train giữ nguyên giá trị gốc

# Tạo file submission chuẩn định dạng Kaggle
submission = pd.DataFrame({
    "Id": range(1461, 1461 + len(test_preds)),
    "SalePrice": test_preds
})

# Lưu trực tiếp vào thư mục exps_/results/
output_path = os.path.join(RES_DIR, "submission_pytorch.csv")
submission.to_csv(output_path, index=False)
print(f"\nĐã xuất file thành công tại: {output_path}")
print(submission.head())

Thư mục chứa dữ liệu: D:\DL_BaiTap-DeTaiThucHanh_Nhom\Tuan4\ml_project\exps_\data\feature1
Thư mục lưu kết quả: D:\DL_BaiTap-DeTaiThucHanh_Nhom\Tuan4\ml_project\exps_\results
Kích thước X_train: (1460, 225) | y_train: (1460,)
Kích thước X_test: (1459, 225)

Bắt đầu huấn luyện mô hình PyTorch MLP...
Epoch [10/50] - Val RMSE: 0.8191
Epoch [20/50] - Val RMSE: 1.1486
Epoch [30/50] - Val RMSE: 1.2091
Epoch [40/50] - Val RMSE: 1.1738
Epoch [50/50] - Val RMSE: 1.1414

Đã xuất file thành công tại: D:\DL_BaiTap-DeTaiThucHanh_Nhom\Tuan4\ml_project\exps_\results\submission_pytorch.csv
     Id  SalePrice
0  1461  10.867966
1  1462   9.513388
2  1463  12.001055
3  1464  12.505196
4  1465  11.932116
